# Understanding Reasoning Models & Test-Time Compute: Insights from DeepSeek-R1
Based on the analysis of architectural shifts, reinforcement learning paradigms, and inference-time scaling principles introduced by modern reasoning models.

This notebook provides code examples, simulations, and algorithmic structures illustrating:
1. **System-1 vs System-2 Generation Latency Profiles**
2. **Group Relative Policy Optimization (GRPO) Objective Function**
3. **Reward Evaluation Mechanisms (Accuracy & Format Checks)**
4. **Inference-Time Scaling & Test-Time Compute (Majority Voting & Self-Correction Simulation)**

## 1. System-1 vs System-2 Generation Profiles
Conventional Large Language Models use direct next-token prediction (System-1 thinking: fast, automatic). Reasoning models (System-2 thinking: slow, analytical) allocate an explicit computational budget during inference to generate structural thinking traces before arriving at a final response.

In [29]:
import time
import numpy as np
import matplotlib.pyplot as plt

def simulate_system1_generation(prompt, processing_speed_tps=50):
    """Simulates a fast direct-response answer."""
    # Typical direct response length
    output_tokens = np.random.randint(50, 150) 
    generation_time = output_tokens / processing_speed_tps
    return {
        "type": "System-1 (Standard LLM)",
        "thinking_tokens": 0,
        "answer_tokens": output_tokens,
        "latency": generation_time
    }

def simulate_system2_generation(prompt, problem_difficulty="hard", processing_speed_tps=40):
    """Simulates a reasoning model generating an internal thinking trace first."""
    if problem_difficulty == "hard":
        thinking_tokens = np.random.randint(800, 2500)
        answer_tokens = np.random.randint(150, 400)
    else:
        thinking_tokens = np.random.randint(200, 600)
        answer_tokens = np.random.randint(100, 200)
        
    total_tokens = thinking_tokens + answer_tokens
    generation_time = total_tokens / processing_speed_tps
    return {
        "type": "System-2 (Reasoning LLM)",
        "thinking_tokens": thinking_tokens,
        "answer_tokens": answer_tokens,
        "latency": generation_time
    }

# Let's run a test simulation
s1_profile = simulate_system1_generation("Solve a complex math problem")
s2_profile = simulate_system2_generation("Solve a complex math problem", problem_difficulty="hard")

print(f"{s1_profile['type']} Profile: Latency = {s1_profile['latency']:.2f}s, Tokens = {s1_profile['answer_tokens']}")
print(f"{s2_profile['type']} Profile: Latency = {s2_profile['latency']:.2f}s, Thinking Tokens = {s2_profile['thinking_tokens']}, Answer Tokens = {s2_profile['answer_tokens']}")

System-1 (Standard LLM) Profile: Latency = 2.30s, Tokens = 115
System-2 (Reasoning LLM) Profile: Latency = 57.67s, Thinking Tokens = 2056, Answer Tokens = 251


## 2. Group Relative Policy Optimization (GRPO)
DeepSeek-R1-Zero uses GRPO instead of traditional PPO. GRPO optimizes policy models without a separate value function network by sampling a group of outputs for each prompt and normalizing rewards against the group mean and standard deviation.

In [30]:
def compute_grpo_advantages(rewards):
    """
    Computes advantages for a group of sampled outputs under GRPO.
    Advantage A_i = (R_i - mean(R)) / (std(R) + epsilon)
    """
    rewards = np.array(rewards, dtype=np.float64)
    mean_r = np.mean(rewards)
    std_r = np.std(rewards)
    
    # Avoid division by zero if all rewards are identical
    if std_r == 0:
        return np.zeros_like(rewards)
        
    advantages = (rewards - mean_r) / (std_r + 1e-8)
    return advantages

# Example tracking 5 candidate answers generated from a single math prompt
sample_rewards = [1.0, 0.0, 1.0, 0.2, 0.5] # Custom evaluation metric scores
grpo_advantages = compute_grpo_advantages(sample_rewards)

for idx, (r, adv) in enumerate(zip(sample_rewards, grpo_advantages)):
    print(f"Sample {idx+1} | Raw Reward: {r:.1f} | GRPO Advantage: {adv:+.4f}")

Sample 1 | Raw Reward: 1.0 | GRPO Advantage: +1.1277
Sample 2 | Raw Reward: 0.0 | GRPO Advantage: -1.3238
Sample 3 | Raw Reward: 1.0 | GRPO Advantage: +1.1277
Sample 4 | Raw Reward: 0.2 | GRPO Advantage: -0.8335
Sample 5 | Raw Reward: 0.5 | GRPO Advantage: -0.0981


## 3. Verifiable Rule-Based Rewards (Accuracy & Format)
To foster reasoning chains without expensive human annotation pipelines, reward rules evaluate strict, deterministic patterns: verifying mathematical precision and confirming the structural presence of `<thought>` and `<answer>` tags.

In [31]:
import re

def evaluate_response(generation_text, ground_truth_answer):
    """
    Computes composite rule-based rewards based on Format and Accuracy.
    """
    reward_format = 0.0
    reward_accuracy = 0.0
    
    # 1. Format Verification: Checking presence of structural tags
    has_thought_start = "<thought>" in generation_text
    has_thought_end = "</thought>" in generation_text
    has_answer_start = "<answer>" in generation_text
    has_answer_end = "</answer>" in generation_text
    
    if has_thought_start and has_thought_end and has_answer_start and has_answer_end:
        # Perfect container format structure
        reward_format = 1.0
    elif has_thought_start or has_answer_start:
        # Partial container format structure
        reward_format = 0.3
        
    # 2. Accuracy Verification: Checking extracted answer
    answer_pattern = r"<answer>(.*?)</answer>"
    match = re.search(answer_pattern, generation_text, re.DOTALL)
    
    if match:
        extracted_content = match.group(1).strip()
        if extracted_content == str(ground_truth_answer):
            reward_accuracy = 1.0
            
    # Return weighted combination or independent components
    return {
        "format_reward": reward_format,
        "accuracy_reward": reward_accuracy,
        "total_reward": reward_format + reward_accuracy
    }

# Test scenarios
correct_sample = """<thought>
The problem asks for 15 * 6.
15 * 6 = (10 * 6) + (5 * 6) = 60 + 30 = 90.
</thought>
<answer>90</answer>"""

malformed_sample = """I figured out that 15 * 6 equals 90 right away."""

print("Correct & Well-Formatted Response Score:", evaluate_response(correct_sample, 90))
print("Malformed Response Score:", evaluate_response(malformed_sample, 90))

Correct & Well-Formatted Response Score: {'format_reward': 1.0, 'accuracy_reward': 1.0, 'total_reward': 2.0}
Malformed Response Score: {'format_reward': 0.0, 'accuracy_reward': 0.0, 'total_reward': 0.0}


## 4. Test-Time Compute & Scaling Properties
As inference compute resource scale expands (e.g., generating more reasoning tokens or applying majority voting / ensembling over multiple thought trajectories), the problem accuracy curve responds non-linearly.

In [32]:
from collections import Counter

def simulate_majority_voting(num_rollouts=16, base_correctness_prob=0.6):
    """
    Simulates test-time scaling through parallel sample rollouts (Majority Voting).
    """
    rollout_results = []
    for _ in range(num_rollouts):
        # Rollout is correct based on its underlying sample probability
        is_correct = np.random.rand() < base_correctness_prob
        rollout_results.append("CorrectAnswer" if is_correct else f"WrongAnswer_{np.random.randint(1,4)}")
        
    # Vote Aggregation
    votes = Counter(rollout_results)
    majority_choice, vote_count = votes.most_common(1)[0]
    
    return {
        "majority_choice": majority_choice,
        "is_successful": majority_choice == "CorrectAnswer",
        "agreement_rate": vote_count / num_rollouts
    }

# Running the test-time scaling metric tracking
print("Single Run Result (N=1):", simulate_majority_voting(num_rollouts=1))
print("Scaled Compute Result (N=16):", simulate_majority_voting(num_rollouts=16))

Single Run Result (N=1): {'majority_choice': 'CorrectAnswer', 'is_successful': True, 'agreement_rate': 1.0}
Scaled Compute Result (N=16): {'majority_choice': 'CorrectAnswer', 'is_successful': True, 'agreement_rate': 0.6875}
